In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
import time as tm
from time import time
import fitsio
import pandas as pd
from galpy.potential import MWPotential2014, vcirc, evaluateRforces
from galpy.orbit import Orbit
from scipy.stats import gaussian_kde, ks_2samp, kstest
from hyppo.ksample import Energy
import corner
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

from astroquery.gaia import Gaia
from astropy.table import Table
from astropy.table import Table, vstack
from glob import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LinearRegression
import os
from datetime import datetime
from matplotlib.ticker import AutoLocator
from pathlib import Path

# Setup and helper functions moved to external module
from equations_file import *

# If you change the external module and want live reload during development, uncomment:
# %load_ext autoreload
# %autoreload 2

In [2]:
with fitsio.FITS(galah_Gaia_fits) as hdul1:
    data_1 = hdul1[1].read()
galah_Gaia_raw = pd.DataFrame(data_1)
galah_Gaia_raw = ensure_native_endian(galah_Gaia_raw)

In [3]:
galah_Gaia = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.2)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['flag_mg_fe'] == 0)
                        # & (galah_Gaia_raw['flag_na_fe'] == 0)
                        # & (galah_Gaia_raw['flag_cu_fe'] == 0)
                        & (galah_Gaia_raw['fe_h'] < -0.75)
                        # & (galah_Gaia_raw['fe_h'] < -0.9)
                        # & (galah_Gaia_raw['ti_fe'] > 0.25)
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
    ]

In [ ]:
NGC3201_galah = pd.merge( galah_Gaia, Vmans_NGC3201_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC5139_galah = pd.merge( galah_Gaia, Vmans_NGC5139_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC1851_galah = pd.merge( galah_Gaia, Vmans_NGC1851_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')

galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)
                                 & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                 & (galah_Gaia_raw['flag_mg_fe'] == 0)
                                 & (galah_Gaia_raw['flag_na_fe'] == 0)
                                 & (galah_Gaia_raw['flag_cu_fe'] == 0)
                                 & ((galah_Gaia_raw['mg_fe'] - galah_Gaia_raw['cu_fe']) - ((1.06923) * galah_Gaia_raw['na_fe']) > 0.40150)
                                 ]

# Kushniruk et al . 2026 https://doi.org/10.1051/0004-6361/202451201
galah_Kushniruk_Thamnos1 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])- 13.5)**2) / (4.0**2)) + (((galah_Gaia['jphi'] + 600)**2) / (350**2)) < 1)
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

galah_Kushniruk_Thamnos2 = galah_Gaia[(((np.sqrt(galah_Gaia['jr'])-8.6)**2) / (3.5**2)) + (((galah_Gaia['jphi'] + 1184)**2) / (175**2)) < 1
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_GSE = galah_Gaia[GSE_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            # & (galah_Gaia['fe_h'] < -0.8)
                            ]

galah_Gaia_Sequoia = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            # & (galah_Gaia['fe_h'] < -0.8)
                            ]

galah_Gaia_Halo = galah_Gaia[(np.sqrt((galah_Gaia['vphi']-230)**2 + (galah_Gaia['vr'])**2 + (galah_Gaia['vz'])**2) > 230) # Koppelman et al. 2019  https://doi.org/10.1051/0004-6361/201936738
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                            # & (galah_Gaia['fe_h'] < -1.0)
                            # & (galah_Gaia['fe_h'] > -1.9)                            
                            # & (galah_Gaia_raw['ti_fe'] > 0.25)
                             ]


In [ ]:
random_5000_1 = galah_Gaia.sample(n=2290)

random_5000_1_mask = (~galah_Gaia['sobject_id'].isin(random_5000_1['sobject_id']))

random_5000_2 = galah_Gaia[random_5000_1_mask].sample(n=90)

In [ ]:
element_list_full_galah = ["h", "fe", "n", "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti" "v", "cr" ,"mn", "co", "ni", "cu", "zn", "rb", "sr", "y", "zr", "mo", "ru", "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.
element_list_partial_small_galah_1 = [ 'fe', 'ca', 'ti', 'na', 'mn', 'cu','mg']
element_list_partial_small_galah_2 = [ 'fe', 'ca', 'na', 'mn', 'cu','mg', 'nd']
element_list_partial_small_galah_3 = [ 'cu', 'ca', 'mn', 'ni','v','na', 'nd']
element_list_partial_small_galah_4 = [ 'cu', 'ca', 'mn', 'ti','v','na', 'nd']
element_list_partial_small_galah_5 = [ 'cu', 'ca', 'mn', 'ti','ba','na', 'nd']
element_list_partial_small_galah_6 = [ 'cu', 'ca', 'mn', 'ti', 'v', 'ba','na', 'nd']
element_list_partial_small_galah_7 = [ 'fe', 'cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_2 = ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_3 = ['fe', 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_4 = ['fe', 'si', 'nd', 'na', 'mg', 'k', 'ca', 'sc', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba']
element_list_partial_small_galah_7_5 = ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_6 = ['v', 'k', 'na', 'mg', 'si', 'ca', 'sc', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_8 = [ 'ca', 'mn', 'y', 'v', 'na', 'nd', 'cr', 'sc']
element_list_partial_small_galah_9 = [ 'cu', 'ca', 'mn', 'ti', 'sc', 'v', 'na', 'nd']
element_list_partial_small_galah_10 = ['fe', 'v', 'sc', 'si', 'ca', 'mn', 'na', 'nd']
element_list_partial_small_galah_12 = ['ca', 'sc', 'na', 'mg', 'si', 'k', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_11 = ['y', 'ba', 'nd', 'eu']


elements_in_Pradosh_Galah_1 = ['ca', 'ti', 'ni','zr', 'ce', 'nd']
elements_in_Pradosh_Galah_2 = ['ca', 'ti', 'ni', 'nd']

element_list = element_list_partial_small_galah_7_5

# element_list_X_frequ = list(X_frequ['element'])
# element_list = element_list_X_frequ

galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia, element_list)

galah_Gaia_new_ratios = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0] # this ontains a full catalog from galah but now it includes ratios for Ca, Si, Ti, Ni, Zr, Ce, and Nd relative to eachother: Pradosh did recomend taking out
                                                                 # Si, and Ni whitch I can do by changing "element_list_partial_large" to element_list_partial_small
galah_gaia_new_element_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]     # this just lists the new columns ratios so I can more easily call them
galah_gaia_new_element_ratio_errors_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[2] 

galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_GSE, element_list)
galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_GSE, element_list)
galah_Gaia_Kushniruk_GSE_new_ratios = galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia, element_list)
galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC3201_galah, element_list)
# galah_Gaia_NGC3201_new_ratios = galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC5139_galah, element_list)
galah_Gaia_NGC5139_new_ratios = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos1, element_list)
# galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos2, element_list)
# galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC1851_galah, element_list)
# galah_Gaia_NGC1851_new_ratios = galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Halo, element_list)
galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# random_5000_1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(random_5000_1, element_list)
# random_5000_1_new_ratios = random_5000_1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# random_5000_2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(random_5000_2, element_list)
# random_5000_2_new_ratios = random_5000_2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

In [ ]:
Halo_Pradosh_galah_Gaia = galah_Gaia_new_ratios[(galah_Gaia_new_ratios['fe_h'] < -0.8)
                                            & (galah_Gaia_new_ratios['ti_fe'] > 0.25)
                                            & (galah_Gaia_new_ratios['Lz'] > -1500)
                                            & (galah_Gaia_new_ratios['Lz'] < 1500)
                                            & (galah_Gaia_new_ratios['ecc_gaia'] > 0.5)
                                            & (galah_Gaia_new_ratios['ecc_gaia'] < 0.8)
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE_new_ratios['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia_new_ratios['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]
# galah_Gaia_Halo_new_ratios = Halo_Pradosh_galah_Gaia


thickdisk_galah_Gaia = galah_Gaia_new_ratios[(galah_Gaia_new_ratios['fe_h'] < -0.8)
                                                 & (galah_Gaia_new_ratios['ti_fe'] > 0.25)
                                                 & (galah_Gaia_new_ratios['Lz'] > 0)
                                                 & (galah_Gaia_new_ratios['ecc_gaia'] < 0.5)
                                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE_new_ratios['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia_new_ratios['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]
print(len(Halo_Pradosh_galah_Gaia))
print(len(thickdisk_galah_Gaia))

In [ ]:
print(len(galah_Gaia_Sequoia_new_ratios))
print(len(galah_Gaia_GSE_new_ratios))
print(len(galah_Gaia_Halo_new_ratios))

In [ ]:
element_list_X_frequ

In [ ]:
''' THESE INCLUDE ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia and Halo vs GSE '''
previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_GSE_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_Halo_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

previos_values_lcation_Halo_vs_GSE = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_Halo_vs_GSE_2d_ks_like_test.csv'
previos_values_lcation_Halo_vs_GSE_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE)


In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia '''
''' fe_h < -0.9'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_fe_minus_0.9_GSE_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_fe_minus_0.9_Halo_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia '''
''' fe_h < -0.75 and ti_fe > 0.25'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_fe_minus_0.75_ti_plus_0.25_GSE_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_fe_minus_0.75_ti_plus_0.25_Halo_vs_Sequoia_2d_ks_like_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia and Halo vs GSE'''
''' fe_h < -0.75 and for Halo ti_fe > 0.25'''
''' Galah Main Bright Faint'''
    
previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_tiforHalo_plus_0.25_Galah_main_faint_bright_GSE_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_tiforHalo_plus_0.25_Galah_main_faint_bright_Halo_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

# previos_values_lcation_Halo_vs_GSE = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_tiforHalo_plus_0.25_Galah_main_faint_bright_Halo_vs_GSE_2d_ks_like_test.csv'
# previos_values_lcation_Halo_vs_GSE_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE)

In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia and Halo vs GSE'''
''' fe_h < -0.75 '''
''' Galah Main Bright Faint'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_main_faint_bright_GSE_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_main_faint_bright_Halo_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

previos_values_lcation_Halo_vs_GSE = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_main_faint_bright_Halo_vs_GSE_2d_ks_test.csv'
previos_values_lcation_Halo_vs_GSE_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE)

In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_Sequoia and Halo vs GSE'''
''' fe_h < -0.75 '''
''' Galah Phase 1 and 2 and Hermes'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_phase1_2_Hermes_GSE_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_phase1_2_Hermes_Halo_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

previos_values_lcation_Halo_vs_GSE = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_phase1_2_Hermes_Halo_vs_GSE_2d_ks_test.csv'
previos_values_lcation_Halo_vs_GSE_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE)


In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Pradosh_Halo_vs_Sequoia and Pradosh_Halo vs GSE'''
''' fe_h < -0.75 '''
''' Galah Phase 1 and 2 and Hermes'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/no_fe_yes_mg_Galah_phase1_2_Hermes_GSE_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/Pradosh_Halo_Galah_phase1_2_Hermes_Halo_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

previos_values_lcation_Halo_vs_GSE = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/Pradosh_Halo_Galah_phase1_2_Hermes_Halo_vs_GSE_2d_ks_test.csv'
previos_values_lcation_Halo_vs_GSE_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE)

In [ ]:
''' THESE INCLUDE ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd'] and 2d '''
''' THAMNOS Stars were removed'''
''' Metal_poor_Halo_vs_Sequoia (Halo with -1.9 < Fe/H < -1.0)'''
''' fe_h < -0.75 '''
''' Galah Phase 1 and 2 and Hermes'''

previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/one_by_one_KS_test_outputs/Metal_poor_Halo_min1.9_to_min1.0Galah_phase1_2_Hermes_Halo_vs_Sequoia_2d_ks_test.csv'
previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

In [ ]:
Top_X = 1000

best_g_vs_s = previos_values_lcation_GSE_vs_Sequoia_raw.sort_values(by='p_value', ascending=True).head(Top_X)
best_h_vs_s = previos_values_lcation_Halo_vs_Sequoia_raw.sort_values(by='p_value', ascending=True).head(Top_X)
best_h_vs_g = previos_values_lcation_Halo_vs_GSE_raw.sort_values(by='p_value', ascending=True).head(Top_X)

# previos_values_lcation_Halo_vs_Sequoia_raw['statistic'] = previos_values_lcation_Halo_vs_Sequoia_raw['statistic'].abs()
# best_h_vs_s = previos_values_lcation_Halo_vs_Sequoia_raw.sort_values(by='statistic', ascending=False).head(Top_X)


# previos_values_lcation_Halo_vs_GSE_raw['statistic'] = previos_values_lcation_Halo_vs_GSE_raw['statistic'].abs()
# best_h_vs_g = previos_values_lcation_Halo_vs_GSE_raw.sort_values(by='statistic', ascending=False).head(Top_X)


In [ ]:
best_h_vs_s.head(60)

In [ ]:
# plt.scatter(galah_Gaia_Sequoia_new_ratios['teff'], galah_Gaia_Sequoia_new_ratios['logg'], c=galah_Gaia_Sequoia_new_ratios['sc_v'], label='Sequoia')
# plt.scatter(galah_Gaia_GSE_new_ratios['teff'], galah_Gaia_GSE_new_ratios['logg'], c=galah_Gaia_GSE_new_ratios['fe_h'], label='GSE')
plt.scatter(galah_Gaia_Halo_new_ratios['teff'], galah_Gaia_Halo_new_ratios['logg'], c=galah_Gaia_Halo_new_ratios['sc_v'], label='Halo')

plt.xlabel('Teff')
plt.ylabel('logg')
plt.colorbar(label='[Fe/H]')
# plt.xlim(3500, 7000)
# plt.ylim(0, 5)
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()

plt.legend()

In [ ]:
match_cols = ['ratio 1', 'ratio 2']

best_g_vs_s_sub = best_g_vs_s.rename(
    columns={
        'p_value': 'p_value_g_s',
        'statistic': 'statistic_g_s',
    }
)

best_h_vs_s_sub = best_h_vs_s.rename(
    columns={
        'p_value': 'p_value_h_s',
        'statistic': 'statistic_h_s',
    }
)

best_h_vs_g_sub = best_h_vs_g.rename(
    columns={
        'p_value': 'p_value_h_g',
        'statistic': 'statistic_h_g',
    }
)

best_all_three = (
    best_g_vs_s_sub
    .merge(best_h_vs_s_sub[match_cols + ['p_value_h_s', 'statistic_h_s']], on=match_cols)
    # .merge(best_h_vs_g_sub[match_cols + ['p_value_h_g', 'statistic_h_g']], on=match_cols)
)

print(f"Found {len(best_all_three)} exact matches across all three best_* DataFrames")
best_all_three_sorted = best_all_three.sort_values(by='p_value_h_s', ascending=True)
# best_all_three_sorted = best_all_three.sort_values(by='statistic_h_s', ascending=False)


display(best_all_three_sorted.head(30))

In [ ]:
top_ratios = ratio_totals_table(best_h_vs_s, galah_gaia_new_element_ratio_list, 2)
# top_ratios = ratio_totals_table(best_all_three_sorted, galah_gaia_new_element_ratio_list, 2)
# top_ratios
top_ratios_no_0 = top_ratios[top_ratios['total'] != 0]
top_ratios_no_0.head(20)



In [ ]:
ratios_of_interest = ['sc_nd', 'si_v']
if isinstance(ratios_of_interest, str):
    ratios_of_interest = [ratios_of_interest]

best_all_three_selected = best_all_three_sorted[
    best_all_three_sorted['ratio 1'].isin(ratios_of_interest)
    | best_all_three_sorted['ratio 2'].isin(ratios_of_interest)
].copy()

print(f"Found {len(best_all_three_selected)} rows where ratio 1 or ratio 2 contains {ratios_of_interest}")
display(best_all_three_selected)

In [ ]:
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.width', 2000,
    'display.max_colwidth', None
):
    display(best_all_three_selected.head(10))

In [ ]:
element_list

In [ ]:
X_frequ = element_frequency(best_h_vs_s['ratio 1'], best_h_vs_s['ratio 2'], element_list)


In [ ]:
gyugyuguy = element_frequency(best_all_three_sorted['ratio 1'], best_all_three_sorted['ratio 2'], element_list)

In [ ]:
plt.figure(figsize=(10, 6))
n_components = 2
elm_ratio_1 = combined_ratio_1
elm_ratio_2 = combined_ratio_2
elm_ratio_3 = 'si_v'

# elm_ratio_1 = 'si_'
# elm_ratio_2 = combined_ratio_2

plt.hist(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna(), bins=30, color = 'g', density=True, histtype='step')
plot_gmm(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna(), f'GSE', n_components = n_components, color='g', maximum_peak=True, shift_right=0.01)
plt.hist(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna(), bins=30, color = 'r', alpha=0.5, label='Sequoia', density=True, histtype='step')
plot_gmm(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna(), f'Sequoia', n_components = n_components, color='r', maximum_peak=True, shift_right=0.10)
plt.hist(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna(), bins=30, color = 'grey', alpha=0.5, label='Halo', density=True, histtype='step')
plot_gmm(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna(), f'Halo', n_components = n_components, color='grey', maximum_peak=True, shift_right=0.19)

# plt.hist(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() +(galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna()), bins=30, color = 'g', density=True, histtype='step')
# plot_gmm(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() + galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna(), f'GSE', n_components = n_components, color='g', maximum_peak=True, shift_right=0.01)
# plt.hist(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna(), bins=30, color = 'r', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna(), f'Sequoia', n_components = n_components, color='r', maximum_peak=True, shift_right=0.10)
# plt.hist(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna(), bins=30, color = 'grey', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna(), f'Halo', n_components = n_components, color='grey', maximum_peak=True, shift_right=0.19)

# plt.hist(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() - (galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna()), bins=30, color = 'g', density=True, histtype='step')
# plot_gmm(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() - galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna(), f'GSE', n_components = n_components, color='g', maximum_peak=True, shift_right=0.01)
# plt.hist(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() - (galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna()), bins=30, color = 'r', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() - galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna(), f'Sequoia', n_components = n_components, color='r', maximum_peak=True, shift_right=0.10)
# plt.hist(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() - (galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna()), bins=30, color = 'grey', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() - galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna(), f'Halo', n_components = n_components, color='grey', maximum_peak=True, shift_right=0.19)

# plt.hist(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() +(galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna()) - galah_Gaia_GSE_new_ratios[elm_ratio_3].dropna(), bins=30, color = 'g', density=True, histtype='step')
# plot_gmm(galah_Gaia_GSE_new_ratios[elm_ratio_1].dropna() + galah_Gaia_GSE_new_ratios[elm_ratio_2].dropna() - galah_Gaia_GSE_new_ratios[elm_ratio_3].dropna(), f'GSE', n_components = n_components, color='g', maximum_peak=True, shift_right=0.01)
# plt.hist(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Sequoia_new_ratios[elm_ratio_3].dropna(), bins=30, color = 'r', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Sequoia_new_ratios[elm_ratio_3].dropna(), f'Sequoia', n_components = n_components, color='r', maximum_peak=True, shift_right=0.10)
# plt.hist(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Halo_new_ratios[elm_ratio_3].dropna(), bins=30, color = 'grey', alpha=0.5, density=True, histtype='step')
# plot_gmm(galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Halo_new_ratios[elm_ratio_3].dropna(), f'Halo', n_components = n_components, color='grey', maximum_peak=True, shift_right=0.19)


plt.xlabel(elm_ratio_1)
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
sample1 = galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna()
sample2 = galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna() 

sample1 = random_5000_1_new_ratios[elm_ratio_2].dropna()
sample2 = random_5000_2_new_ratios[elm_ratio_2].dropna() 

# sample1 = galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna()
# sample2 = galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna()

# sample1 = galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() - galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna()
# sample2 = galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() - galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna()

# sample1 = [random.gauss(0.4, 0.3) for _ in range(600)]     # (mean, sd)
# sample2 = [random.gauss(0.45, 0.3) for _ in range(2500)]
# sample1 = [5,5,5,5,5,5,3,3,3,3,3,3,3,4,4,4,4,4,4,4,4,2,2,2,2,2,2,2]
# sample2 = [8,8,8,8,8,8,8,8,8,9,9,9,9,9,9,9,9,9,7,7,7,7,7,7,7,7,7,7,7,7,7,9,9,9,9,8,9,9,7,9,8,9,9,8,9,9,8,7,8,7,8]

# sample1 = galah_Gaia_Sequoia_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Sequoia_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Sequoia_new_ratios[elm_ratio_3].dropna()
# sample2 = galah_Gaia_Halo_new_ratios[elm_ratio_1].dropna() + galah_Gaia_Halo_new_ratios[elm_ratio_2].dropna() - galah_Gaia_Halo_new_ratios[elm_ratio_3].dropna()

# statistic, p_value = ks_2samp(sample1, sample2)
statistic, p_value = kstest(sample1, sample2)


print(f'p-value: {p_value}    and   statistic: {statistic}')

# p-value: 6.0486402029175425e-09    and   statistic: 0.3417238511124974 # for v_ca - si_v for Halo vs Sequoia
# p-value: 6.452861280819423e-09    and   statistic: 0.3411728009981285 # for v_ca + v_ti - si_v for Halo vs Sequoia
# p-value: 2.987588534009413e-08    and   statistic: 0.32780203784570594 # for v_ca + v_ti for Halo vs Sequoia

In [ ]:
plt.hist(sample1, bins=30, color = 'r', alpha=0.5, density=True, histtype='step', label='Sample 1')
plt.hist(sample2, bins=30, color = 'g', alpha=0.5, density=True, histtype='step', label='Sample 2')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# ratio1 = elm_ratio_1
# ratio2 = elm_ratio_2

ratio_1 = 'cr_na'
ratio_2 = 'nd_y'

ratio1 = 'si_v'
ratio2 = 'cr_ca'
ratio3 = 'ni_k'
ratio4 = 'ca_v'
ratio5 = 'si_v'
ratio6 = 'co_ca'

# ratio1 = 'na_sc'
# ratio2 = 'mn_sc'
# ratio3 = 'nd_sc'
# ratio4 = 'ca_v'
# ratio5 = 'si_v'
# ratio6 = 'co_k'


# ratio1 = 'k_v'   # these were good after ading a ti cut but before doinging only galah main faint and bright
# ratio2 = 'ca_v'
# ratio3 = 'si_v'
# ratio4 = 'mn_sc'
# ratio5 = 'nd_sc'
# ratio6 = 'na_sc'

# ratio1 = 'si_v'   #these were good untill I added a ti cut, but dont include ti_v
# ratio2 = 'ca_v'
# ratio3 = 'ti_v'
# ratio4 = 'mn_sc'
# ratio5 = 'nd_sc'
# ratio6 = 'co_sc'

combined_ratio_1 = f'{ratio1}_min_{ratio2}'#_p_{ratio3}'
combined_ratio_2 = f'{ratio4}_p_{ratio5}_min_{ratio6}'

# ratio_1 = combined_ratio_1
# ratio_2 = combined_ratio_2

galah_Gaia_Sequoia_new_ratios[combined_ratio_1] = (
    galah_Gaia_Sequoia_new_ratios[ratio1]
    - galah_Gaia_Sequoia_new_ratios[ratio2]
    # + galah_Gaia_Sequoia_new_ratios[ratio3]
)
galah_Gaia_Halo_new_ratios[combined_ratio_1] = (
    galah_Gaia_Halo_new_ratios[ratio1]
    - galah_Gaia_Halo_new_ratios[ratio2]
    # + galah_Gaia_Halo_new_ratios[ratio3]
)
galah_Gaia_GSE_new_ratios[combined_ratio_1] = (
    galah_Gaia_GSE_new_ratios[ratio1]
    - galah_Gaia_GSE_new_ratios[ratio2]
    # + galah_Gaia_GSE_new_ratios[ratio3]
)
# random_5000_1_new_ratios[combined_ratio_1] = (
#     random_5000_1_new_ratios[ratio1]
#     + random_5000_1_new_ratios[ratio2]
#     # + random_5000_1_new_ratios[ratio3]
# )
# random_5000_2_new_ratios[combined_ratio_1] = (
#     random_5000_2_new_ratios[ratio1]
#     + random_5000_2_new_ratios[ratio2]
#     # + random_5000_2_new_ratios[ratio3]
# )

galah_Gaia_Sequoia_new_ratios[combined_ratio_2] = (
    galah_Gaia_Sequoia_new_ratios[ratio4]
    + galah_Gaia_Sequoia_new_ratios[ratio5]
    - galah_Gaia_Sequoia_new_ratios[ratio6]
)
galah_Gaia_Halo_new_ratios[combined_ratio_2] = (
    galah_Gaia_Halo_new_ratios[ratio4]
    + galah_Gaia_Halo_new_ratios[ratio5]
    - galah_Gaia_Halo_new_ratios[ratio6]
)
galah_Gaia_GSE_new_ratios[combined_ratio_2] = (
    galah_Gaia_GSE_new_ratios[ratio4]
    + galah_Gaia_GSE_new_ratios[ratio5]
    - galah_Gaia_GSE_new_ratios[ratio6]
)
# random_5000_1_new_ratios[combined_ratio_2] = (
#     random_5000_1_new_ratios[ratio4]
#     + random_5000_1_new_ratios[ratio5]
#     + random_5000_1_new_ratios[ratio6]
# )
# random_5000_2_new_ratios[combined_ratio_2] = (
#     random_5000_2_new_ratios[ratio4]
#     + random_5000_2_new_ratios[ratio5]
#     + random_5000_2_new_ratios[ratio6]
# )
# ratio1 = combined_ratio_1
# ratio2 = combined_ratio_2

# convert to numpy arrays
# X = galah_Gaia_Sequoia_new_ratios[[ratio_1, ratio_2]].dropna().to_numpy()
X = galah_Gaia_Halo_new_ratios[[ratio_1, ratio_2]].dropna().to_numpy()
Y = galah_Gaia_GSE_new_ratios[[ratio_1, ratio_2]].dropna().to_numpy()

# X = random_5000_1_new_ratios[[ratio_1, ratio_2]].dropna().to_numpy()
# Y = random_5000_2_new_ratios[[ratio_1, ratio_2]].dropna().to_numpy()

# multivariate comparison
statistic, p_value = Energy().test(X, Y)

print(f'statistic: {statistic}, p-value: {p_value},  for {ratio_1} vs {ratio_2}')

# print(f'std({ratio_1}: {np.sqrt(np.mean(galah_Gaia_Sequoia_new_ratios[f'e_{ratio1}']))})')

#statistic: 0.06060561706032123, p-value: 1.395893565671733e-07 for v_ca - si_v +v_ti vs sc_nd for Halo vs Sequoia
#statistic: 0.04472336651462405, p-value: 1.673203426050091e-07,  for v_ca - si_v vs sc_nd for Halo vs Sequoia
# statistic: 0.08047511120134651, p-value: 9.735948606821972e-09,  for si_v_p_ca_v_p_ti_v vs nd_sc_p_ni_sc_p_cr_sc for Halo vs Sequoia
# statistic: 0.07689870349001003, p-value: 3.589844497769417e-07,  for si_v_p_ca_v_p_ti_v vs co_sc_p_ni_sc_p_cu_sc
# statistic: 0.08286302185982213, p-value: 1.0005983922384975e-08,  for si_v_p_ca_v_p_ti_v vs mn_sc_p_nd_sc_p_ni_sc for Halo vs Sequoia
# statistic: 0.08334057791423806, p-value: 5.691440903738436e-09,  for si_v_p_ca_v_p_ti_v vs mn_sc_p_nd_sc_p_cr_sc
# statistic: 0.08466773438993347, p-value: 4.62436405279537e-09,  for si_v_p_ca_v_p_ti_v vs cr_sc_p_nd_sc_p_co_sc for Halo vs Sequoia
# statistic: 0.0877544490406221, p-value: 4.089869223613862e-09,  for si_v_p_ca_v_p_ti_v vs mn_sc_p_nd_sc_p_co_sc
# statistic: 0.0771478703759406, p-value: 2.6788577906919025e-09,  for si_v_p_ca_v vs mn_sc_p_nd_sc_p_co_sc    
# # statistic: 0.08200447852197862, p-value: 2.349661636997737e-09,  for ca_v_p_si_v vs mn_sc_p_nd_sc_p_na_sc   # everything for this and above this had Mg ignored
# statistic: 0.07292351828132654, p-value: 4.6515275032901415e-08,  for si_v_p_ca_v vs mn_sc_p_nd_sc_p_co_sc    # everything for this and below this had Mg flaged and a ti cut
# statistic: 0.07836248946872154, p-value: 4.136422316140312e-08,  for ca_v_p_si_v vs mn_sc_p_nd_sc_p_na_sc


# statistic: 0.08527826660830382, p-value: 2.8063238909707747e-05,  for k_v_p_ca_v vs mn_sc_p_nd_sc_p_na_sc   # These have galah main faith and bright but with a Ti cut on the halo
# statistic: 0.0991820592124768, p-value: 2.1740469032962485e-05,  for k_v_p_ca_v_p_si_v vs mn_sc_p_nd_sc_p_na_sc
# statistic: 0.04445584946945647, p-value: 1.6457678605223252e-05,  for ca_v vs co_k
# statistic: 0.09265807529219144, p-value: 1.3426535541811343e-05,  for ca_v_p_co_k vs zn_na_p_sc_na
# statistic: 0.07472658568180597, p-value: 5.5459205829045465e-06,  for ca_v_p_co_k vs zn_na

# statistic: 0.04940182024706924, p-value: 4.346552497575713e-06,  for ca_v vs co_k # These have NO ti cut, but include only galah main faint and bright
# statistic: 0.07034981948694927, p-value: 6.817474378635082e-06,  for co_k_p_cr_k vs ca_v
# statistic: 0.0662305728989379, p-value: 2.6180775394116626e-06,  for co_k vs ca_v_p_si_v
# statistic: 0.10043153859711508, p-value: 8.200275299825618e-07,  for na_sc_min_ni_sc vs ca_v_p_si_v_p_co_k
# statistic: 0.09157532228477511, p-value: 2.9370097368114493e-06,  for nd_sc vs ca_v_p_si_v_min_co_k
# statistic: 0.10803984991929703, p-value: 7.620434340697612e-07,  for nd_sc_min_ni_sc vs ca_v_p_si_v_min_co_k
# statistic: 0.1148900931315928, p-value: 6.603214565314754e-07,  for na_sc_p_mn_sc vs ca_v_p_si_v_min_co_k
# statistic: 0.10853603054696123, p-value: 5.119960536610066e-07,  for na_sc vs ca_v_p_si_v_min_co_k  # i knd of like this spesific combination a little more than the one below this
# statistic: 0.12754707462108952, p-value: 3.3057368987261775e-07,  for nd_sc_p_mn_sc vs ca_v_p_si_v_min_co_k
# statistic: 0.13428028829926997, p-value: 2.3667492480129469e-07,  for nd_sc_p_mn_sc_p_na_sc vs ca_v_p_si_v_min_co_k



In [ ]:
galah_Gaia_Sequoia_new_ratios[f'e_{combined_ratio_1}'] = ( np.sqrt(
    (galah_Gaia_Sequoia_new_ratios[f'e_{ratio1}'])**2
    + (galah_Gaia_Sequoia_new_ratios[f'e_{ratio2}'])**2
    # + (galah_Gaia_Sequoia_new_ratios[f'e_{ratio3}'])**2
))

galah_Gaia_Sequoia_new_ratios[f'e_{combined_ratio_2}'] = (np.sqrt(
    (galah_Gaia_Sequoia_new_ratios[f'e_{ratio4}'])**2
    + (galah_Gaia_Sequoia_new_ratios[f'e_{ratio5}']))**2
    + (galah_Gaia_Sequoia_new_ratios[f'e_{ratio6}'])**2
)

print(np.mean(galah_Gaia_Sequoia_new_ratios[f'e_{combined_ratio_1}']))

In [ ]:
list_in_combined_ratios = [ratio1, ratio2, ratio4, ratio5, ratio6, combined_ratio_1, combined_ratio_2]

# for ratio in list_in_combined_ratios:
#         plt.figure(figsize=(10, 6))
#         mean_err = np.mean(galah_Gaia_Sequoia_new_ratios[f'e_{ratio}'])
#         plt.hist(galah_Gaia_Sequoia_new_ratios[f'e_{ratio}'], bins = 20)
#         plt.title(f"Sequoia errors for {ratio}: mean = {mean_err}")

for ratio in list_in_combined_ratios:
        plt.figure(figsize=(10, 6))
        median_err = np.median(galah_Gaia_Sequoia_new_ratios[f'e_{ratio}'])
        plt.hist(galah_Gaia_Sequoia_new_ratios[f'e_{ratio}'], bins = 20)
        plt.title(f"Sequoia errors for {ratio}: mean = {median_err}")


# print(f'{ratio1}: {np.mean(galah_Gaia_Sequoia_new_ratios[ratio1])}')
# print(f'{ratio2}: {np.mean(galah_Gaia_Sequoia_new_ratios[ratio2])}')
# print(f'{ratio4}: {np.mean(galah_Gaia_Sequoia_new_ratios[ratio4])}')
# print(f'{ratio5}: {np.mean(galah_Gaia_Sequoia_new_ratios[ratio5])}')
# print(f'{ratio6}: {np.mean(galah_Gaia_Sequoia_new_ratios[ratio6])}')

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))

elm_ratio1 = 'sc_fe' #this is pretty good
elm_ratio2 = 'si_v'

elm_ratio1 = 'cr_na' #best for GSE vs Sequoia
elm_ratio2 = 'nd_y'

elm_ratio1 = 'eu_ba' ####  This might be good for separating the GSE and Sequoia

elm_ratio1 = 'cr_sc'
elm_ratio2 = 'ca_v'

# elm_ratio2 = 'nd_y'
# elm_ratio1 = 'mn_na'

# elm_ratio1 = ratio_1
# elm_ratio2 = ratio_2

# elm_ratio1 = 'na_fe'
# elm_ratio2 = 'mn_mg'
# elm_ratio3 = 'si_v'
# elm_ratio4 = 'si_v'

# elm_ratio1 = 'cr_na'
# elm_ratio2 = 'nd_y'
# elm_ratio3 = 'si_v'
# elm_ratio4 = 'si_mn'


all_x_values = []
all_y_values = []

# plt.scatter(galah_Gaia_new_ratios[elm_ratio1], galah_Gaia_new_ratios[elm_ratio2], color = 'grey', label = 'All Galah', s = 0.3, alpha = 0.2)
# sns.kdeplot(
#     x=galah_Gaia_new_ratios[elm_ratio1].dropna(),
#     y=galah_Gaia_new_ratios[elm_ratio2].dropna(),
#     levels=8,
#     cmap='Greys',
#     thresh=0.05,
#     fill=False,
#     linewidths=1.5,
#     cut=0,
#     alpha=0.9,
#     ax=ax,
# )

sc  = ax.scatter(galah_Gaia_Halo_new_ratios[elm_ratio1], galah_Gaia_Halo_new_ratios[elm_ratio2],
            c = (galah_Gaia_Halo_new_ratios['fe_h']), cmap = 'rainbow',s = 10,
            #  color = 'grey', s = 40,
             label = 'Halo', alpha = 0.9)
# sns.kdeplot(
#     # x=(galah_Gaia_Halo_new_ratios[elm_ratio1]+ galah_Gaia_Halo_new_ratios[elm_ratio3]).dropna(),
#     x= galah_Gaia_Halo_new_ratios[elm_ratio1].dropna(),
#     # y =(galah_Gaia_Halo_new_ratios[elm_ratio2]+ galah_Gaia_Halo_new_ratios[elm_ratio4]).dropna(),
#     y= galah_Gaia_Halo_new_ratios[elm_ratio2].dropna(),
#     levels=8,
#     cmap='cool',
#     thresh=0.05,
#     fill=False,
#     linewidths=1.5,
#     cut=0,
#     alpha=0.9,
#     ax=ax,
#     label = 'Halo'
# )

# sc = ax.scatter(galah_Gaia_GSE_new_ratios[elm_ratio1], galah_Gaia_GSE_new_ratios[elm_ratio2], 
#                 # c = galah_Gaia_GSE_new_ratios['y_fe'], cmap = 'rainbow',
#                 color = 'green', 
#                 label = 'GSE', s = 40)
# sns.kdeplot(
#     # x=(galah_Gaia_GSE_new_ratios[elm_ratio1]+ galah_Gaia_GSE_new_ratios[elm_ratio3]).dropna(),
#     x= galah_Gaia_GSE_new_ratios[elm_ratio1].dropna(),
#     # y =(galah_Gaia_GSE_new_ratios[elm_ratio2]+ galah_Gaia_GSE_new_ratios[elm_ratio4]).dropna(),
#     y= galah_Gaia_GSE_new_ratios[elm_ratio2].dropna(),
#     levels=8,
#     cmap='Greens',
#     thresh=0.05,
#     fill=False,
#     linewidths=1.5,
#     cut=0,
#     alpha=0.9,
#     ax=ax,
#     label = 'GSE'
# )

num = len(galah_Gaia_Sequoia_new_ratios)

sc =ax.scatter(galah_Gaia_Sequoia_new_ratios[elm_ratio1], galah_Gaia_Sequoia_new_ratios[elm_ratio2], 
        #    c = galah_Gaia_Sequoia_new_ratios['fe_h'], cmap = 'rainbow',
           color = 'k', 
           label = f'Sequoia  N: {num}', s = 30)
# sns.kdeplot(
#     # x=(galah_Gaia_Sequoia_new_ratios[elm_ratio1]+ galah_Gaia_Sequoia_new_ratios[elm_ratio3]).dropna(),
#         x=galah_Gaia_Sequoia_new_ratios[elm_ratio1].dropna(),
#     # y =(galah_Gaia_Sequoia_new_ratios[elm_ratio2]+ galah_Gaia_Sequoia_new_ratios[elm_ratio4]).dropna(),
#     y= galah_Gaia_Sequoia_new_ratios[elm_ratio2].dropna(),
#     levels=8,
#     cmap='Reds',
#     thresh=0.05,
#     fill=False,
#     linewidths=1.5,
#     cut=0,
#     alpha=0.9,
#     ax=ax,
#     label = 'Sequoia'
# )

# ax.scatter(v_mean + 0.02, sc_mean - 0.14, color = 'k', s = 40)
# all_x_values.append(galah_Gaia_Sequoia_new_ratios[elm_ratio1])
# all_y_values.append(galah_Gaia_Sequoia_new_ratios[elm_ratio2])

ax.set_xlabel(F'{elm_ratio1}')
ax.set_ylabel(F'{elm_ratio2}')
leg = ax.legend(fontsize=10, labelspacing=0.8, borderpad=1.2)
ax.legend()
for handle in leg.legend_handles:
    handle.set_sizes([30])
    ax.set_title(F'Galah {elm_ratio2} vs {elm_ratio1}')

# plt.colorbar(sc)

In [ ]:
# galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios[galah_Gaia_Sequoia_new_ratios['y_fe'] < 0.8]
# plt.hist(galah_Gaia_Sequoia_new_ratios['y_fe'], bins = 30)
# plt.scatter(galah_Gaia_Halo_new_ratios['fe_h'], galah_Gaia_Halo_new_ratios['y_fe'])
# plt.scatter(galah_Gaia_Sequoia_new_ratios['fe_h'], galah_Gaia_Sequoia_new_ratios['y_fe'])
# galah_Gaia_Sequoia_new_ratios

# ratio = 'na_sc'
ratio = combined_ratio_1
# plt.hist(galah_Gaia_GSE_new_ratios[ratio], color = 'green', bins = 30, density = True, linewidth = 0.5, histtype='step',)
plt.hist(galah_Gaia_Sequoia_new_ratios[ratio], color = 'red', bins = 30, density = True, linewidth = 0.5, histtype='step',)
plt.hist(galah_Gaia_Halo_new_ratios[ratio], color = 'grey', bins = 30, density = True, linewidth = 0.5, histtype='step',)

In [ ]:
ratio_list = galah_gaia_new_element_ratio_list
# ratio_list = peak_table['ratio']
# ratio_list = ['fe_h']
# ratio_list = ['nd_y', 'mn_si', 'nd_si']
# ratio_list = ['sc_v', 'v_ca', 'sc_na', 'si_v', 'co_sc']
# ratio_list = ['co_sc', 'sc_na', 'cr_k', 'sc_ni', 'cr_sc', 'si_cu', 'sc_cu', 'co_k', 'si_na', 'sc_mn', 'k_ni', 'co_si', 'k_na', 'k_mn', 'si_ni', 'k_cu', 'sc_ti', 'co_zn', 'sc_ba', 'sc_ba', 'na_zn', 'na_ti', 'si_mn', 'cr_cu', 'zn_mn', 'co_ti', 'ti_cu', 'nd_na']
# ratio_list = one_d_test_test_rndom_5000['ratio'][0:15]
# ratio_list = top_ratios['ratio']
# ratio_list = list_in_combined_ratios


n_components = 1
maximum_peak = True

object_list = {
                'GSE': (galah_Gaia_GSE_new_ratios, 'g', 1, 0.5, 0.02),  #df, color, linewidth, alpha, shift_right
                'Sequoia': (galah_Gaia_Sequoia_new_ratios, 'r', 1, 0.5, 0.10),
                'Halo': (galah_Gaia_Halo_new_ratios, 'grey', 1, 0.5, 0.18),
                # 'ran_1': (random_5000_1_new_ratios, 'blue', 1, 0.5, 0.18),
                # 'ran_2': (random_5000_2_new_ratios, 'r', 1, 0.5, 0.1),
}

key_list = ['ratio']
for key in object_list:
    key_list.append(key)
print(key_list)
peak_table = pd.DataFrame(columns=key_list)  ##### call this out if you dont want the peak_table


for ratio in ratio_list:
    plt.figure(figsize=(10, 6))
    all_values = []
    ratio_peaks = [ratio]

    for key, (df, color, lw, alpha, shift_right) in object_list.items():

        values = df[ratio].to_numpy()
        values = values[np.isfinite(values)]
        all_values.append(values)

        df_copy = df
        num_stars = str(len(df_copy))
        local_peak = plot_gmm(df_copy[ratio] , f'{key}: N = {num_stars}', color, n_components, maximum_peak = maximum_peak, shift_right = shift_right)
        ratio_peaks.append(local_peak)
        plt.hist(df[ratio], density = True, bins  = 30, histtype='step', facecolor = 'none', alpha = alpha, edgecolor=color, linewidth = lw)
        # mean = np.round(np.mean(df[ratio]), 3)
        # plt.axvline(x=mean, color=color, linestyle='--')

    peak_table.loc[len(peak_table)] = ratio_peaks  ##### call this out if you dont want the peak_table

    all_values = np.concatenate(all_values)
    low = np.percentile(all_values, 0)
    high = np.percentile(all_values, 100)

    plt.xlim(low, high)

    # plt.title(F'Galah {ratio} distribution')
    plt.title(f"{ratio}: H and S diff by {np.round(abs(peak_table['Halo'].iloc[-1] - peak_table['Sequoia'].iloc[-1]),3)} dex")    ##### call this out if you dont want the peak_table
    plt.xlabel(f'[{ratio}]')
    plt.ylabel('density')
    plt.legend()
peak_table['H_S_diff'] = abs(peak_table['Halo'] - peak_table['Sequoia'])   ##### call this out if you dont want the peak_table
# peak_table['H_S_sd'] = abs(peak_table['Halo'] - peak_table['Sequoia'])
peak_table = peak_table.sort_values(by = 'H_S_diff', ascending = False)    ##### call this out if you dont want the peak_table
display(peak_table.sort_values(by = 'H_S_diff', ascending = False))        ##### call this out if you dont want the peak_table

In [ ]:
def plot_gmm_copy(data, label, color, n_components, maximum_peak = False, plot_compents = False, shift_right = 0.01, shift_up = 0.99):
    # """Plot GMM curve"""
    # x, pdf = fit_gmm_and_get_pdf(data, n_components)
    # plt.plot(x, pdf, lw=2, color=color, label=f"{label}")

    """Plot GMM curve + components"""

    x, pdf, gmm = fit_gmm_and_get_pdf(data, n_components)

    # Total mixture
    plot = plt.plot(x, pdf, lw=2, color=color, label=label)

    if maximum_peak:
        max_index = np.argmax(pdf)
        max_x = np.round(x[max_index][0], 3)
        plt.text(shift_right, shift_up, f' L:{max_x}', transform=plt.gca().transAxes, color=color, ha='left', va='top')
        return(max_x)
    # Individual components
    components = sorted(
    zip(gmm.weights_, gmm.means_, gmm.covariances_),
    key=lambda x: x[1][0]
    )
    for i, (weight, mean, cov) in enumerate(components):

        sigma = np.sqrt(cov[0][0])

        component_pdf = weight * norm.pdf(
            x.flatten(),
            mean[0],
            sigma
        )
        if plot_compents:
            plt.plot(
            x.flatten(),
            component_pdf,
            color=color,
            ls='--',
            lw=1.5,
            alpha=0.7
            )

            if maximum_peak == False:
                print(mean)
                mean_text = np.round(mean[0], 3)
                plt.text(shift_right, 0.98 - 0.05*i, f'C: {mean_text}', transform=plt.gca().transAxes, color=color, ha='left', va='top')

In [ ]:
plt.figure(figsize=(10, 6))

s_peak_copy = plot_gmm_copy(galah_Gaia_Sequoia_new_ratios['fe_h'], 'sequoia', 'red', 2, maximum_peak = True, shift_right = 0.10)
h_peak_copy = plot_gmm_copy(galah_Gaia_Halo_new_ratios['fe_h'], 'sequoia', 'grey', 2, maximum_peak=True, shift_right = 0.18)
g_peak_copy = plot_gmm_copy(galah_Gaia_GSE_new_ratios['fe_h'], 'gse', 'green', 2, maximum_peak=True, shift_right = 0.02)

In [ ]:
for i in range(len(peak_table)):
    print(peak_table['ratio']) 

In [ ]:
ratio = 'ti_fe'
plt.hist(galah_Gaia_Sequoia_new_ratios[ratio], density = True, bins  = 30, histtype='step', facecolor = 'none', alpha = alpha, edgecolor='red', linewidth = lw)
plt.hist(galah_Gaia_GSE_new_ratios[ratio], density = True, bins  = 30, histtype='step', facecolor = 'none', alpha = alpha, edgecolor='green', linewidth = lw)
plt.hist(galah_Gaia_Halo_new_ratios[ratio], density = True, bins  = 30, histtype='step', facecolor = 'none', alpha = alpha, edgecolor=color, linewidth = lw)

In [ ]:
plt.scatter(galah_Gaia_Halo_new_ratios['jphi'], galah_Gaia_Halo_new_ratios['energy'], color = 'grey')
plt.scatter(galah_Gaia_Sequoia_new_ratios['jphi'], galah_Gaia_Sequoia_new_ratios['energy'], 
            c = galah_Gaia_Sequoia_new_ratios['d_from_combined_ratio_mean'], cmap = 'rainbow', 
            # vmin = -1.1, vmax = -0.2
            vmin = 0.05, vmax = 0.3
            )
cbar = plt.colorbar()
cbar.set_label('d_from_combined_ratio_mean')

In [ ]:
v_mean = np.mean(galah_Gaia_Sequoia_new_ratios[combined_ratio_1]) + 0.02
sc_mean = np.mean(galah_Gaia_Sequoia_new_ratios[combined_ratio_2]) - 0.14

In [ ]:
galah_Gaia_Sequoia_new_ratios['d_from_combined_ratio_mean'] = np.sqrt((galah_Gaia_Sequoia_new_ratios[combined_ratio_1]-v_mean)**2 + (galah_Gaia_Sequoia_new_ratios[combined_ratio_2]-sc_mean)**2)